# The `bench2` Range–Doppler benchmark

The synthetic dataset behind this project's CFAR-vs-deep-learning comparison. This notebook covers
**what the data is, how it was built, what it looks like, how hard it is, and what evidence supports
trusting it** — no detector results.

**All data here is synthetic.** It is generated by this project's own FMCW simulator. The simulator
was configured to **reproduce the published CARRADA Range–Doppler grid geometry** — the same map
dimensions and bin spacings — so that a target occupies a comparable number of cells. It shares
**nothing else** with CARRADA: not its scenes, its clutter, its waveform, or its measurements.

Run top to bottom. Nothing here modifies data.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from radar.config import RadarConfig, derive_radar_params
from radar.simulator import make_axes

CFG = RadarConfig()
DERIVED = derive_radar_params(CFG)
_, _, RANGE_AXIS, VEL_AXIS = make_axes(CFG)

SPLITS = ["bench2_train", "bench2_val", "bench2_tune", "bench2_test", "bench2_snr_sweep"]
MAIN = ["bench2_train", "bench2_val", "bench2_tune", "bench2_test"]

# One scenario order everywhere in the notebook, so the reader never has to re-orient.
ORDER = ["isolated", "masking", "unresolved", "clutter", "clutter_edge", "multi", "mixed", "noise"]

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
BLUE, ORANGE, AQUA, PURPLE = "#2a78d6", "#eb6834", "#1baf7a", "#8172b3"
SPLIT_COLOR = dict(zip(MAIN, [BLUE, ORANGE, PURPLE, AQUA]))

plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
                     "axes.edgecolor": "#c3c2b7", "grid.color": GRID,
                     "text.color": INK, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "font.size": 10, "figure.dpi": 110})


def tidy(ax):
    ax.grid(axis="y", linewidth=0.8)
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)


def load_meta(split):
    m = pd.read_csv(ROOT / "data" / split / "metadata.csv")
    m["split"] = split
    return m


META = pd.concat([load_meta(s) for s in SPLITS], ignore_index=True)
print("ready")

---
## 0 · The dataset at a glance

In [ ]:
n_scen = META[META.split != "bench2_snr_sweep"].condition.nunique()
glance = [
    ("RD map size", f"{CFG.N // 2} range x {CFG.M} velocity bins"),
    ("Range span", f"0 to {DERIVED.range_max:.1f} m"),
    ("Velocity span", f"+/-{DERIVED.vel_max:.2f} m/s"),
    ("Range bin spacing", f"{DERIVED.range_res:.3f} m"),
    ("Velocity bin spacing", f"{DERIVED.vel_res:.3f} m/s"),
    ("Splits", f"{len(SPLITS)} ({', '.join(s.replace('bench2_', '') for s in SPLITS)})"),
    ("Total frames", f"{len(META):,}"),
    ("Total labelled targets", f"{int(META.num_targets.sum()):,}"),
    ("Scenario types", f"{n_scen}"),
    ("Saved per frame", "RD magnitude map, complex RD map, label mask, labels.csv, metadata.json"),
]
display(pd.DataFrame(glance, columns=["", "value"]).style.hide(axis="index"))

Every frame is one coherent processing interval turned into a Range–Doppler map, with a
per-target label giving its true range, velocity, bin indices and cell SNR.

---
## 1 · Radar and RD-map geometry

The simulator was configured to reproduce the published **CARRADA Range–Doppler grid geometry**, so
a target spreads over a comparable number of cells. Secondary waveform parameters are in the
appendix.

In [ ]:
rows = [
    ("Carrier frequency", f"{CFG.fc / 1e9:g} GHz"),
    ("Bandwidth", f"{CFG.B / 1e6:g} MHz"),
    ("Range resolution", f"{DERIVED.range_res:.3f} m"),
    ("Velocity resolution", f"{DERIVED.vel_res:.3f} m/s"),
    ("Max range", f"{DERIVED.range_max:.2f} m"),
    ("Max velocity", f"+/-{DERIVED.vel_max:.2f} m/s"),
    ("RD map", f"{CFG.N // 2} x {CFG.M}"),
]
display(pd.DataFrame(rows, columns=["parameter", "value"]).style.hide(axis="index"))

### What one frame looks like

Every RD figure in this notebook uses the same conventions, so this one is worth reading carefully.

In [ ]:
_INDEX = {}


def frame_index(split):
    """condition -> [(num_targets, frame_dir)], built once per split.

    Read from metadata.csv rather than by opening every frame's metadata.json: the same lookup
    used to cost a full 3,200-file scan on each call, and this notebook needs it a dozen times.
    """
    if split not in _INDEX:
        # `frame_id` in metadata.csv is the directory NAME ("frame_000000"), not an integer.
        m = load_meta(split).set_index("frame_id")
        idx = {}
        for fd in sorted((ROOT / "data" / split / "frames").iterdir()):
            if fd.name not in m.index:
                continue
            row = m.loc[fd.name]
            idx.setdefault(row.condition, []).append((int(row.num_targets), fd))
        if not idx:
            raise RuntimeError(f"frame index for {split} is empty -- metadata.csv and the frames/ "
                               f"directory do not line up")
        _INDEX[split] = idx
    return _INDEX[split]


def frames_of(split, cond):
    """All frame dirs of a condition, with their target count."""
    return frame_index(split).get(cond, [])


def pick(split, cond):
    """One representative frame: the one whose target count is closest to typical."""
    fr = frames_of(split, cond)
    if not fr:
        return None
    typical = int(np.median([n for n, _ in fr]))
    return min(fr, key=lambda t: abs(t[0] - typical))[1]


def rd_db(fd):
    """RD magnitude as dB above THAT frame's own noise floor.

    Not normalised by the map maximum: a target near the detection threshold sits only a few dB
    above the brightest noise, and peak normalisation would hide it entirely.
    """
    mag = np.load(fd / "rd_map_mag.npy")
    return 10 * np.log10(mag ** 2 / np.median(mag ** 2) + 1e-12)


def labels_of(fd):
    p = fd / "labels.csv"
    return pd.read_csv(p) if p.stat().st_size > 5 else pd.DataFrame()


EXT = [VEL_AXIS[0], VEL_AXIS[-1], RANGE_AXIS[0], RANGE_AXIS[-1]]

# A STRONG isolated target, chosen so the anatomy is legible. Difficulty across the whole dataset
# is section 6's job; this figure only has to teach the axes and the colour scale.
cands = frames_of("bench2_test", "isolated")[:400]
best = max(cands, key=lambda t: labels_of(t[1]).cell_snr_db.max())
fd = best[1]
db, lab = rd_db(fd), labels_of(fd)
r0, v0 = float(lab.range_m.iloc[0]), float(lab.velocity_mps.iloc[0])

fig, ax = plt.subplots(figsize=(9.5, 6.8))
im = ax.imshow(db, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=30, extent=EXT)
ax.scatter(v0, r0, s=420, facecolors="none", edgecolors="white", linewidths=2.0)

# Keep annotations inside the axes: text placed outside gets clipped by tight_layout.
side = -1 if v0 > 0 else 1
ax.annotate("labelled target (ground truth)",
            xy=(v0, r0), xytext=(v0 + side * 4.5, r0 + 10),
            color="white", fontsize=10.5,
            ha="right" if side < 0 else "left",
            arrowprops=dict(arrowstyle="->", color="white", lw=1.4))
ax.text(-12.8, 3.5, "background: receiver noise", color="white", fontsize=9.5, alpha=0.9)
print(f"frame {fd.name}: target at {r0:.1f} m, {v0:+.2f} m/s, "
      f"cell SNR {lab.cell_snr_db.iloc[0]:.1f} dB")
ax.set_xlabel("Velocity (m/s)     negative = approaching")
ax.set_ylabel("Range (m)")
ax.set_title("Anatomy of one frame - `isolated` scenario, one target", loc="left", fontsize=12)
ax.grid(False)
fig.colorbar(im, ax=ax, label="dB above the frame's own noise floor", pad=0.02)
fig.tight_layout()
plt.show()

*<span style='color:#6b6b6b'>The bright spot is the target; the white ring is its ground-truth label. Colour is decibels above that frame's own noise floor, so 0 is typical noise and a target must rise clearly above it. A strong target is shown here so the layout is legible — most targets are far fainter, which is what §6 quantifies. Every later RD figure uses these same axes and this same colour scale.</span>*

---
## 2 · Dataset construction and splits

Five datasets, each generated with a **different random seed**, so no frame can appear twice.

In [ ]:
ROLE = {
    "bench2_train": ("network training", "no"),
    "bench2_val": ("all development decisions", "YES"),
    "bench2_tune": ("CFAR window geometry", "YES"),
    "bench2_test": ("final reported numbers", "read once, at the end"),
    "bench2_snr_sweep": ("Pd-vs-SNR diagnostic", "no"),
}
sp = (META.groupby("split")
      .agg(frames=("frame_id", "size"), targets=("num_targets", "sum"))
      .reindex(SPLITS))
sp["purpose"] = [ROLE[s][0] for s in sp.index]
sp["used for decisions?"] = [ROLE[s][1] for s in sp.index]
sp.index = [s.replace("bench2_", "") for s in sp.index]
display(sp)

counts = (META[META.split != "bench2_snr_sweep"]
          .pivot_table(index="condition", columns="split", values="frame_id",
                       aggfunc="size", fill_value=0))
counts = counts.reindex([c for c in ORDER if c in counts.index])
counts.columns = [c.replace("bench2_", "") for c in counts.columns]
print()
print("frames per scenario (exact -- the generator cycles through the named scenarios):")
display(counts)

Two design choices worth stating:

**`tune` exists so the classical baseline could be fitted without touching evaluation data.** The
CFAR window geometry was selected on `tune`; because §5 shows `tune` is distributed like the others,
that choice transfers without having been made on data the detectors are later scored on.

**`unresolved` is absent from `train` on purpose.** Both target types sit inside one resolution
cell, so there is nothing learnable there; it is kept in `val` and `test` as a sanity check.

---
## 3 · Scenario design

Each scenario is a scene type built to stress a *specific* detector weakness rather than to be
generically hard.

In [ ]:
SCEN = [
    ("isolated", "clean-case reference", "1", "no",
     "none - the easy baseline every method should handle"),
    ("masking", "a strong return hides a weak neighbour", "2 (strong + weak)", "no",
     "the pair sits inside one reference window, so the weak target inflates its own noise estimate"),
    ("unresolved", "two targets inside one resolution cell", "2 (0.05-0.35 m apart)", "no",
     "physically unresolvable - separation is below the measured resolution limit"),
    ("clutter", "target inside a clutter ridge", "1-3, |v| < 1.2 m/s", "yes, CNR 8-25 dB",
     "heavy-tailed clutter breaks the homogeneous-noise assumption a fixed threshold relies on"),
    ("clutter_edge", "target beside a clutter ridge", "1-2, |v| 2.5-4.2 m/s", "yes, CNR 12-25 dB",
     "the target is in clean noise but its reference window reaches into the ridge"),
    ("multi", "several well-separated targets", "4-5", "no",
     "mild mutual interference; tests behaviour as scene density rises"),
    ("mixed", "everything at once", "0-5", "yes, CNR 5-22 dB",
     "clutter plus multipath plus variable density - the closest to an uncontrolled scene"),
    ("noise", "no targets at all", "0", "no",
     "measures the false-alarm rate; the only scenario with no Pd"),
]
display(pd.DataFrame(SCEN, columns=["scenario", "purpose", "targets", "clutter",
                                    "main difficulty"]).style.hide(axis="index"))

Three of the eight (`masking`, `unresolved`, `clutter_edge`) are deliberately adversarial. That
is intentional design, and it is also why a single pooled score over this dataset would not describe
a typical scene — per-scenario reporting is the point.

---
## 4 · What the scenarios look like

One representative frame per scenario, chosen by rule (the frame whose target count is closest to
typical for that scenario), not by eye.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 8.4))
for ax, cond in zip(axes.ravel(), ORDER):
    fd = pick("bench2_test", cond)
    lab = labels_of(fd)
    im = ax.imshow(rd_db(fd), aspect="auto", origin="lower", cmap="viridis",
                   vmin=0, vmax=30, extent=EXT)
    if len(lab):
        ax.scatter(lab.velocity_mps, lab.range_m, s=150, facecolors="none",
                   edgecolors="white", linewidths=1.6)
    ax.set_title(f"{cond}   ({len(lab)} target{'' if len(lab) == 1 else 's'})",
                 loc="left", fontsize=11)
    ax.grid(False)
    ax.tick_params(labelsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("Range (m)")
for ax in axes[-1, :]:
    ax.set_xlabel("Velocity (m/s)")
fig.colorbar(im, ax=axes, label="dB above the frame's noise floor", pad=0.012, aspect=40)
fig.suptitle("One frame per scenario - white rings are labelled targets", x=0.006, ha="left",
             fontsize=13)
plt.show()

*<span style='color:#6b6b6b'>Compare the backgrounds: `isolated`, `masking`, `unresolved` and `multi` sit in clean noise, while `clutter`, `clutter_edge` and `mixed` carry a bright horizontal ridge near zero velocity. `noise` has no target at all.</span>*

### The four hard cases, zoomed

The full-map view is too coarse to show what makes these scenarios difficult. Each panel is a small
window around the targets.

In [ ]:
def zoom_panel(ax, fd, half_r=3.0, half_v=3.0, title=""):
    """Zoom on a frame's targets, in physical units.

    Centred on the STRONGEST target, not the mean of all of them: `clutter` carries 1-3 targets
    that can sit far apart, and their centroid may contain no target at all.
    """
    lab = labels_of(fd)
    db = rd_db(fd)
    if len(lab):
        anchor = lab.loc[lab.cell_snr_db.idxmax()]
        rc, vc = float(anchor.range_m), float(anchor.velocity_mps)
    else:
        rc, vc = RANGE_AXIS[len(RANGE_AXIS) // 2], 0.0

    ri = np.clip(np.searchsorted(RANGE_AXIS, [rc - half_r, rc + half_r]), 0, len(RANGE_AXIS) - 1)
    vi = np.clip(np.searchsorted(VEL_AXIS, [vc - half_v, vc + half_v]), 0, len(VEL_AXIS) - 1)

    sub = db[ri[0]:ri[1] + 1, vi[0]:vi[1] + 1]
    ext = [VEL_AXIS[vi[0]], VEL_AXIS[vi[1]], RANGE_AXIS[ri[0]], RANGE_AXIS[ri[1]]]
    im = ax.imshow(sub, aspect="auto", origin="lower", cmap="viridis",
                   vmin=0, vmax=30, extent=ext, interpolation="nearest")
    shown = 0
    for _, L in lab.iterrows():
        if ext[0] <= L.velocity_mps <= ext[1] and ext[2] <= L.range_m <= ext[3]:
            ax.scatter(L.velocity_mps, L.range_m, s=300, facecolors="none",
                       edgecolors="white", linewidths=2.2)
            shown += 1
    ax.set_title(title, loc="left", fontsize=10)
    ax.grid(False)
    ax.tick_params(labelsize=8)
    assert shown, f"no target visible in the {title!r} zoom"
    return im


# Tight on the pair for the first two; wider for the clutter pair, to show target against ridge.
PANELS = [
    ("masking", 2.6, 2.2, "masking" + chr(10) + "strong + weak partner"),
    ("unresolved", 1.2, 1.2, "unresolved" + chr(10) + "both in one resolution cell"),
    ("clutter", 6.0, 6.0, "clutter" + chr(10) + "target INSIDE the ridge"),
    ("clutter_edge", 6.0, 7.0, "clutter_edge" + chr(10) + "target BESIDE the ridge"),
]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.8))
for ax, (cond, hr, hv, title) in zip(axes, PANELS):
    im = zoom_panel(ax, pick("bench2_test", cond), hr, hv, title)
    ax.set_xlabel("Velocity (m/s)")
axes[0].set_ylabel("Range (m)")
fig.colorbar(im, ax=axes, label="dB above noise floor", pad=0.012, aspect=30)
plt.show()

*<span style='color:#6b6b6b'>Left two: both labelled targets are ringed — in `masking` the weaker one is a faint smear beside the strong return, and in `unresolved` the two rings sit on what is visually a single blob. Right two: the same clutter ridge, with the target inside it and then just outside it.</span>*

---
## 5 · Dataset coverage

*Where are the targets, and are the splits comparable?* If `val` or `tune` were easier or harder
than `test`, every decision made on them would be quietly biased.

In [ ]:
def targets_of(split, n_sample=1600, seed=0):
    """Target-level rows from a RANDOM sample of frames.

    Random, not evenly spaced: the generator assigns scenarios cyclically, so an evenly-spaced
    sample aliases against that cycle and can miss whole scenarios.
    """
    fds = sorted((ROOT / "data" / split / "frames").iterdir())
    if n_sample and n_sample < len(fds):
        idx = np.random.default_rng(seed).choice(len(fds), n_sample, replace=False)
        fds = [fds[i] for i in sorted(idx)]

    rows = []
    for fd in fds:
        lab = labels_of(fd)
        if not len(lab):
            continue
        m = json.loads((fd / "metadata.json").read_text(encoding="utf-8"))
        cl = m.get("clutter", {})
        lab = lab.copy()
        lab["condition"] = m.get("condition", "sweep")
        lab["frame"] = fd.name
        lab["split"] = split
        lab["clutter_cnr"] = cl.get("cnr_db") if cl.get("enabled") else np.nan
        lab["clutter_shape"] = cl.get("shape") if cl.get("enabled") else np.nan
        rows.append(lab)
    return pd.concat(rows, ignore_index=True)


TARGETS = pd.concat([targets_of(s) for s in MAIN], ignore_index=True)
T = TARGETS[TARGETS.split == "bench2_test"]
print(f"sampled {len(TARGETS):,} targets across {len(MAIN)} splits "
      f"({len(T):,} from test)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (col, label) in zip(axes, [("cell_snr_db", "Cell SNR (dB)"),
                                   ("range_m", "Range (m)"),
                                   ("abs_v", "|Velocity| (m/s)")]):
    for s in MAIN:
        d = TARGETS[TARGETS.split == s]
        v = d.velocity_mps.abs() if col == "abs_v" else d[col]
        v = np.sort(v.values)
        ax.plot(v, np.arange(1, len(v) + 1) / len(v), lw=2,
                color=SPLIT_COLOR[s], label=s.replace("bench2_", ""))
    ax.set_xlabel(label)
    ax.set_ylabel("cumulative fraction of targets")
    tidy(ax)
axes[0].legend(frameon=False, fontsize=9, loc="lower right")
fig.suptitle("Split comparability - the four curves should be indistinguishable",
             x=0.006, ha="left", fontsize=12.5)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

*<span style='color:#6b6b6b'>Cumulative distributions rather than overlaid histograms: four step-curves on one axis are readable where four histograms are not. Any split that was systematically easier would show as a curve shifted right on cell SNR.</span>*

In [ ]:
comp = pd.DataFrame({
    "frames": [int((META.split == s).sum()) for s in MAIN],
    "median cell SNR (dB)": [TARGETS.loc[TARGETS.split == s, "cell_snr_db"].median() for s in MAIN],
    "mean range (m)": [TARGETS.loc[TARGETS.split == s, "range_m"].mean() for s in MAIN],
    "median range (m)": [TARGETS.loc[TARGETS.split == s, "range_m"].median() for s in MAIN],
    "mean |velocity| (m/s)": [TARGETS.loc[TARGETS.split == s, "velocity_mps"].abs().mean()
                              for s in MAIN],
    "mean targets/frame": [META.loc[META.split == s, "num_targets"].mean() for s in MAIN],
}, index=[s.replace("bench2_", "") for s in MAIN]).round(2)
display(comp)

spread = (comp.drop(columns="frames").max() - comp.drop(columns="frames").min())
print("largest difference between any two splits:")
print(spread.round(3).to_string())

`tune` is included deliberately: it selected the CFAR window geometry, so it has to be
comparable to `test` for that choice to transfer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})

ax = axes[0]
hb = ax.hexbin(T.velocity_mps, T.range_m, gridsize=28, cmap="viridis", mincnt=1)
# Axes span the WHOLE map, not just the occupied part: the unused margin is the point.
ax.set_xlim(VEL_AXIS[0], VEL_AXIS[-1])
ax.set_ylim(RANGE_AXIS[0], RANGE_AXIS[-1])
for lo, hi, axis in ((T.range_m.min(), T.range_m.max(), "y"),
                     (T.velocity_mps.min(), T.velocity_mps.max(), "x")):
    (ax.axhline if axis == "y" else ax.axvline)(lo, color=ORANGE, ls=":", lw=1.3)
    (ax.axhline if axis == "y" else ax.axvline)(hi, color=ORANGE, ls=":", lw=1.3)
ax.set_xlabel("Velocity (m/s)")
ax.set_ylabel("Range (m)")
ax.set_title("Targets occupy a bounded part of the field of view", loc="left", fontsize=11.5)
ax.grid(False)
fig.colorbar(hb, ax=ax, label="targets per cell", pad=0.02)

ax = axes[1]
tpf = META[META.split == "bench2_test"].num_targets
vals, cnts = np.unique(tpf, return_counts=True)
ax.bar(vals, 100 * cnts / cnts.sum(), color=BLUE, width=0.65)
for v, c in zip(vals, cnts):
    ax.text(v, 100 * c / cnts.sum() + 0.6, f"{100 * c / cnts.sum():.0f}%",
            ha="center", fontsize=8.5, color=MUTED)
ax.set_xlabel("targets in a frame")
ax.set_ylabel("% of frames")
ax.set_xticks(vals)
ax.set_title("Frames are not all single-target", loc="left", fontsize=11.5)
tidy(ax)
fig.tight_layout()
plt.show()

print(f"targets per frame: mean {tpf.mean():.2f}, median {int(tpf.median())}, "
      f"max {int(tpf.max())};  {100 * (tpf == 0).mean():.0f}% of frames have none")

*<span style='color:#6b6b6b'>Left: dotted lines mark where targets actually fall. They are spread near-uniformly over 8-45 m and +/-8 m/s, while the map itself spans 0-50 m and +/-13.4 m/s — so the outer margin of the field of view carries no targets and is never exercised. Right: frame occupancy runs from empty to five targets, so results must not be read as single-target performance.</span>*

---
## 6 · Target difficulty

*How hard are the targets?* Cell SNR is the difficulty dial — measured per target from a clean map,
so it already includes the range law and scalloping rather than being assumed from the request.

In [ ]:
order = [c for c in ORDER if c in set(T.condition) and c != "noise"]
data = [T.loc[T.condition == c, "cell_snr_db"].values for c in order]

fig, ax = plt.subplots(figsize=(11, 4.6))
bp = ax.boxplot(data, patch_artist=True, showfliers=False, widths=0.6)
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels(order)
for patch in bp["boxes"]:
    patch.set_facecolor(BLUE)
    patch.set_alpha(0.35)
    patch.set_edgecolor(BLUE)
for part in ("whiskers", "caps", "medians"):
    for line in bp[part]:
        line.set_color(BLUE)
        line.set_linewidth(1.5)
ax.set_ylabel("Cell SNR (dB)")
ax.set_title("Difficulty by scenario (test split)", loc="left", fontsize=12)
tidy(ax)
fig.tight_layout()
plt.show()

*<span style='color:#6b6b6b'>The paired scenarios (`masking`, `unresolved`) sit lowest because each contains a deliberately weak partner; `multi` and `mixed` follow the default range law. The spread within each box is the range law acting across the field of view.</span>*

### Against the detection transition

In [ ]:
s = T.cell_snr_db.values

fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.hist(s, bins=60, color=BLUE, edgecolor=SURFACE, linewidth=0.4)
for x, lab in ((4.8, "Pd 0.1"), (10.9, "Pd 0.5"), (19.4, "Pd 0.9")):
    ax.axvline(x, color=ORANGE, linewidth=1.6, linestyle="--")
    ax.text(x, ax.get_ylim()[1] * 0.94, f" {lab}", color=ORANGE, fontsize=9)
ax.set_xlabel("Cell SNR (dB)")
ax.set_ylabel("targets")
ax.set_title("Target difficulty spans the detector transition region", loc="left", fontsize=12)
tidy(ax)
fig.tight_layout()
plt.show()

print(f"median {np.median(s):.1f} dB    below 5 dB: {100 * np.mean(s < 5):.1f}%    "
      f"above 30 dB: {100 * np.mean(s >= 30):.1f}%")

*<span style='color:#6b6b6b'>Dashed lines mark where a textbook detector at Pfa 1e-4 reaches Pd 0.1, 0.5 and 0.9. Targets far to the right are found by anything and discriminate nothing; the useful mass sits across the transition.</span>*

---
## 7 · What the dataset deliberately varies

Two quantities are drawn per frame and **recorded**, rather than fixed. That makes them measured
variables, so an ablation is a grouping of existing results rather than a separate experiment. Each
one sweeps a known detector failure mode:

* **pair separation** in `masking` — sweeps contamination of the reference window, since a close
  neighbour falls inside the region used to estimate noise;
* **clutter strength and tail shape** — sweeps the homogeneity assumption, which a fixed threshold
  depends on;
* **`unresolved` separation** — sweeps the physical resolution limit itself.

In [ ]:
mask = T[T.condition == "masking"]
sep = mask.groupby("frame").range_m.apply(lambda g: abs(g.max() - g.min()))
unres = T[T.condition == "unresolved"]
sep_u = unres.groupby("frame").range_m.apply(lambda g: abs(g.max() - g.min()))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1, 1.15]})

ax = axes[0]
ax.hist(sep, bins=28, color=BLUE, edgecolor=SURFACE, linewidth=0.4, label="masking")
ax.hist(sep_u, bins=14, color=ORANGE, edgecolor=SURFACE, linewidth=0.4, alpha=0.85,
        label="unresolved")
ax.set_xlabel("pair separation in range (m)")
ax.set_ylabel("frames")
ax.set_title("Separation is swept, not fixed", loc="left", fontsize=11.5)
ax.legend(frameon=False, fontsize=9)
tidy(ax)

ax = axes[1]
cl = T.dropna(subset=["clutter_cnr"])
sc = ax.scatter(cl.clutter_cnr, cl.clutter_shape, c=cl.cell_snr_db, cmap="viridis",
                s=14, alpha=0.75)
ax.set_xlabel("clutter-to-noise ratio (dB)   ->  stronger clutter")
ax.set_ylabel("Weibull shape   ->  lower = spikier tail")
ax.set_title("Clutter coverage: strength and tail severity together", loc="left", fontsize=11.5)
ax.grid(True, linewidth=0.6, alpha=0.5)
ax.set_axisbelow(True)
for sp_ in ("top", "right"):
    ax.spines[sp_].set_visible(False)
fig.colorbar(sc, ax=ax, label="target cell SNR (dB)", pad=0.02)
fig.tight_layout()
plt.show()

print(f"masking separation     {sep.min():.2f} - {sep.max():.2f} m")
print(f"unresolved separation  {sep_u.min():.2f} - {sep_u.max():.2f} m")
print(f"clutter CNR            {cl.clutter_cnr.min():.1f} - {cl.clutter_cnr.max():.1f} dB")
print(f"Weibull shape          {cl.clutter_shape.min():.2f} - {cl.clutter_shape.max():.2f}")

*<span style='color:#6b6b6b'>Left: the two paired scenarios tile the separation axis — `unresolved` from 0.05 to 0.35 m, `masking` from 0.41 to 2.49 m — so the dataset covers both sides of the measured resolution limit, with only a narrow band between them unsampled. Right: the clutter frames fill the strength x tail-shape plane rather than clustering, so the two can be analysed independently afterwards.</span>*

The physical resolution limit is **measured, not assumed** — `scripts/measure_psf.py` derives
it directly from the point-spread function, and the separation ablation reports detection against
it. This notebook only shows that the dataset *covers* both sides of that limit.

---
## 8 · Class imbalance

Target cells are rare. This matters for one reason: **pixel accuracy is not a usable metric here.**
A model predicting "no target" everywhere would score over 99.8% correct.

In [ ]:
rows = []
for split in ["bench2_train", "bench2_val", "bench2_test"]:
    fds = sorted((ROOT / "data" / split / "frames").iterdir())
    idx = np.random.default_rng(0).choice(len(fds), 150, replace=False)
    pos = tot = 0
    for i in idx:
        m = np.load(fds[i] / "label_mask.npy")
        pos += int(m.sum())
        tot += m.size
    rows.append(dict(split=split.replace("bench2_", ""),
                     positive_rate=pos / tot,
                     background_rate=1 - pos / tot,
                     ratio=f"1 : {int(tot / pos):,}"))
imb = pd.DataFrame(rows).set_index("split")
display(imb.style.format({"positive_rate": "{:.5f}", "background_rate": "{:.5f}"}))

Roughly one target cell per thousand background cells, consistently across splits.

---
## 9 · Dataset validation

Full gates live in `scripts/verify_splits.py` (is anything broken?) and `scripts/audit_datasets.py`
(is it what the configs asked for?). Their results:

In [ ]:
checks = [
    ("No leakage between splits", "PASS", "zero shared frames, compared by content hash rather "
                                          "than by seed"),
    ("No non-finite values", "PASS", "zero NaN or Inf across all maps"),
    ("Targets inside the testable region", "PASS", "zero targets in the untested border"),
    ("Coordinate -> bin conversion", "PASS", "1047/1047 labels land on the expected bin"),
    ("Scenario counts match the configs", "PASS", "all eight scenarios, exact counts"),
    ("High-SNR isolated alignment", "PASS", "100% on isolated, multi and the SNR sweep"),
    ("Interference displaces the peak", "EXPECTED", "see the alignment analysis below"),
]
display(pd.DataFrame(checks, columns=["check", "result", "detail"]).style.hide(axis="index"))

### The one nuance: label-to-peak alignment

The label records the target's **physical location**. The brightest cell nearby is a different
thing: noise, a neighbouring target, clutter or multipath can all pull the local maximum off the
true position. So **100% alignment should not be expected**, and its absence in hard scenarios is
the dataset working as designed rather than a labelling error.

Measured below with the same definition the verification script uses: for targets at **≥ 15 dB**
cell SNR, is the maximum of the surrounding 5×5 window **within 1 bin** of the label?

In [ ]:
def alignment(split, n_frames=1600, seed=1, snr_floor=15.0):
    """Peak-to-label alignment, per scenario. Same definition as scripts/verify_splits.py."""
    fds = sorted((ROOT / "data" / split / "frames").iterdir())
    idx = np.random.default_rng(seed).choice(len(fds), min(n_frames, len(fds)), replace=False)
    rows = []
    for i in sorted(idx):
        fd = fds[i]
        lab = labels_of(fd)
        if not len(lab):
            continue
        m = json.loads((fd / "metadata.json").read_text(encoding="utf-8"))
        mag = np.load(fd / "rd_map_mag.npy")
        for _, L in lab.iterrows():
            rb, vb = int(L["range_bin"]), int(L["velocity_bin"])
            w = mag[max(0, rb - 2):rb + 3, max(0, vb - 2):vb + 3]
            pk = np.unravel_index(int(np.argmax(w)), w.shape)
            err = max(abs(pk[0] - min(rb, 2)), abs(pk[1] - min(vb, 2)))
            rows.append(dict(condition=m.get("condition", "?"),
                             snr=float(L["cell_snr_db"]), aligned=err <= 1))
    d = pd.DataFrame(rows)
    return d[d.snr >= snr_floor]


AL = alignment("bench2_test")
per = (AL.groupby("condition")
       .agg(targets=("aligned", "size"), aligned=("aligned", "mean")))
per["aligned %"] = (100 * per.aligned).round(1)
per = per.drop(columns="aligned").reindex([c for c in ORDER if c in per.index])
display(per)
print(f"overall: {100 * AL.aligned.mean():.1f}%   "
      f"(n={len(AL):,} targets at >= 15 dB, from a 1,600-frame sample of bench2_test)")

*<span style='color:#6b6b6b'>Alignment is near-perfect in clean scenarios and falls exactly where interference is the point of the scenario. That ordering is the evidence that the labels are correct and the peak simply moves.</span>*

**Independent confirmation.** Regenerating the misaligned frames with noise and clutter
switched off puts every label back exactly on the peak (`scripts/verify_splits.py`). The labels are
right; the brightest cell is just not always the target when something else competes.

### What this validation cannot show

Every frame comes from **one simulator**. These checks establish that the dataset is internally
consistent and matches its configuration — they say nothing about whether results obtained on it
transfer to real radar measurements. That limitation is a property of the study design, not
something further validation of this dataset could remove.

---
## Appendix

### A.1 Full radar parameters

In [ ]:
rows = [
    ("Carrier frequency", f"{CFG.fc / 1e9:g} GHz"),
    ("Bandwidth", f"{CFG.B / 1e6:g} MHz"),
    ("Chirp duration", f"{CFG.Tc * 1e6:g} us"),
    ("Samples per chirp (N)", CFG.N),
    ("Chirps per frame (M)", CFG.M),
    ("Sampling frequency", f"{DERIVED.fs / 1e6:.4f} MHz"),
    ("Frame time", f"{CFG.M * CFG.Tc * 1e3:.3f} ms"),
    ("Range resolution", f"{DERIVED.range_res:.4f} m"),
    ("Max range", f"{DERIVED.range_max:.2f} m"),
    ("Velocity resolution", f"{DERIVED.vel_res:.4f} m/s"),
    ("Max velocity", f"+/-{DERIVED.vel_max:.2f} m/s"),
]
display(pd.DataFrame(rows, columns=["parameter", "value"]).style.hide(axis="index"))

### A.2 What is stored per frame

`rd_map_mag.npy` discards phase. The complex map is kept alongside it so a magnitude-vs-I/Q input
ablation remains possible without regenerating the dataset.

In [ ]:
fd = sorted((ROOT / "data" / "bench2_test" / "frames").iterdir())[0]
print("files per frame:")
for f in sorted(fd.iterdir()):
    print(f"   {f.name}")
print()
print("labels.csv columns:", ", ".join(pd.read_csv(fd / "labels.csv").columns))

### A.3 Per-scenario detail

In [ ]:
per_frame = META[META.split == "bench2_test"].groupby("condition").num_targets.mean()
detail = (T.groupby("condition")
          .agg(targets=("cell_snr_db", "size"),
               snr_median=("cell_snr_db", "median"),
               snr_min=("cell_snr_db", "min"),
               snr_max=("cell_snr_db", "max"),
               range_min=("range_m", "min"),
               range_max=("range_m", "max"),
               absvel_max=("velocity_mps", lambda v: v.abs().max()),
               clutter=("clutter_cnr", lambda v: f"{100 * v.notna().mean():.0f}%"))
          .round(2))
detail["tgt/frame"] = [round(per_frame.get(c, np.nan), 2) for c in detail.index]
display(detail.reindex([c for c in ORDER if c in detail.index]))

---
*Rebuild with `python scripts/build_dataset_overview_nb.py`. Every number is computed from the
dataset at run time; nothing is pasted.*